# Lab 25 — MCP server from scratch

> ⏱ 90-110 min · 🟡 Intermediate

Build a working MCP server end-to-end. By the end of this notebook you'll have:
- a notes server exposing 3 tools, 2 resources, 1 prompt template
- tested it with the MCP Inspector (visual) and a Python client (programmatic)
- walked through 4 schema-inference failure modes with concrete fixes
- deployed it over Streamable HTTP with token-based auth

**Prerequisites read first**: [MCP foundations](../../concepts/tools/mcp-foundations.md) + [Building an MCP server](../../concepts/tools/building-an-mcp-server.md).

This notebook follows the 9-step structure from the lab README. Each step is self-contained; you can re-run any step independently.

## Step 0 — Environment setup

Verify `fastmcp` is installed (it's in the project's `pyproject.toml`). FastMCP 3.0+ is required; this lab uses FastMCP 3.x features (decorator-based registration, type-hint schema inference, the `Client` context manager, OpenTelemetry hooks).

If the import fails or the version is below 3.0, install with `pip install --upgrade 'fastmcp>=3.0'`.

In [ ]:
import sys
import subprocess
from pathlib import Path

try:
    import fastmcp
    print(f"fastmcp version: {fastmcp.__version__}")
    major = int(fastmcp.__version__.split(".")[0])
    if major < 3:
        print("⚠ Lab requires FastMCP 3.x. Run: pip install --upgrade 'fastmcp>=3.0'")
    else:
        print("✓ FastMCP 3.x available")
except ImportError:
    print("✗ fastmcp not installed. Run: pip install 'fastmcp>=3.0'")
    sys.exit(1)

LAB_DIR = Path.cwd()
print(f"Lab working directory: {LAB_DIR}")

## Step 1 — Build the three tools

We'll build a notes server with three CRUD tools: `create_note`, `get_note`, `list_notes`. The minimum useful surface; `delete_note` and `update_note` are stretch goals (Step 8).

Notice the three FastMCP conventions:
1. **Decorator-based registration.** `@mcp.tool()` makes a Python function into an MCP tool. No JSON Schema by hand.
2. **Type hints become schemas.** `title: str` and `body: str` map to JSON Schema `{"type": "string"}` automatically.
3. **Docstrings become tool descriptions.** The first line is the tool's short description; the full docstring goes into the schema. **Write good docstrings** — the LLM uses them to decide whether to call your tool.

We'll write the server to a file (`notes_server.py`) so we can run it as a subprocess for the MCP Inspector workflow in Step 4.

In [ ]:
SERVER_PATH = LAB_DIR / "notes_server.py"

SERVER_CODE = """\
\"\"\"Lab 25 notes server — MCP server with 3 tools, 2 resources, 1 prompt.\"\"\"
from fastmcp import FastMCP

mcp = FastMCP("notes-server")

# In-memory storage. Production: swap for SQLite / PostgreSQL.
NOTES: dict[str, str] = {}


@mcp.tool()
def create_note(title: str, body: str) -> dict:
    \"\"\"Create a note with the given title and body.

    Args:
        title: The note title; must be unique. Returns status='exists' if duplicate.
        body: The note body. No length limit.

    Returns:
        A dict with 'status' (created|exists) and 'title'.
    \"\"\"
    if title in NOTES:
        return {"status": "exists", "title": title}
    NOTES[title] = body
    return {"status": "created", "title": title}


@mcp.tool()
def get_note(title: str) -> dict:
    \"\"\"Retrieve a note by exact title match.

    Args:
        title: The exact note title to look up.

    Returns:
        A dict with 'status' (ok|not_found), 'title', and 'body' if found.
    \"\"\"
    if title not in NOTES:
        return {"status": "not_found", "title": title}
    return {"status": "ok", "title": title, "body": NOTES[title]}


@mcp.tool()
def list_notes() -> list[str]:
    \"\"\"List all note titles currently in the server.

    Returns:
        A list of note titles. Empty list if no notes exist.
    \"\"\"
    return sorted(NOTES.keys())


# Resources and prompt added in later steps
"""

SERVER_PATH.write_text(SERVER_CODE)
print(f"Wrote {SERVER_PATH} ({SERVER_PATH.stat().st_size} bytes)")

## Step 2 — Add the two resources

Resources are the **host-controlled, read-only** primitive. The host (Claude Desktop, your agent) decides when to fetch them — not the LLM. Use resources when:
- The data is read-only by definition
- The host has structural knowledge that the data is relevant (e.g., "always include the project README in the context")

The two resources we'll add:
- **`notes://all`** — a static URI returning all notes concatenated. The host can pull this whenever it wants to ground the LLM with all available notes.
- **`notes://{title}`** — a URI-templated resource where the title is a path parameter. The host can resolve `notes://Q3-plan` to get just that note's body.

The templated-URI pattern is what makes resources scale to large catalogs — you don't need one resource per note; you have one templated resource and the host fills in the title.

In [ ]:
RESOURCES_BLOCK = """\

@mcp.resource("notes://all")
def all_notes() -> str:
    \"\"\"All notes concatenated as a single text blob, separated by horizontal rules.

    Read this resource when you need the full notes corpus in the context.
    \"\"\"
    if not NOTES:
        return "[no notes yet]"
    return "\\n\\n---\\n\\n".join(
        f"# {title}\\n\\n{body}" for title, body in sorted(NOTES.items())
    )


@mcp.resource("notes://{title}")
def one_note(title: str) -> str:
    \"\"\"A single note's body, addressed by URI template.

    Resolves notes://Q3-plan to the body of the note titled 'Q3-plan'.
    Returns a placeholder if the title is not found.
    \"\"\"
    return NOTES.get(title, "[note not found]")

"""

# Append to the server file
existing = SERVER_PATH.read_text()
# Insert before the "# Resources and prompt added in later steps" marker
new_content = existing.replace(
    "# Resources and prompt added in later steps\n",
    RESOURCES_BLOCK + "# Prompt added in next step\n",
    1,
)
SERVER_PATH.write_text(new_content)
print(f"Updated {SERVER_PATH} ({SERVER_PATH.stat().st_size} bytes)")
print("\n--- Last 30 lines of server file: ---")
print("\n".join(SERVER_PATH.read_text().splitlines()[-30:]))

## Step 3 — Add the prompt template

Prompts are the **user-controlled** primitive. The user invokes them (typically as slash commands in the host UI like Claude Desktop or Cursor). Parameters in the prompt function signature become slash-command parameters.

We'll add `summarize_note(title)` — a parametric template the user invokes as `/summarize_note title="Q3-plan"`. The prompt fetches the note body and constructs a summarization instruction the LLM then executes.

This separates **template authorship** (server-side, you write it once) from **template use** (user-side, the user picks it from a menu). The user doesn't have to remember the prompt's exact wording; the user just picks the slash command.

In [ ]:
PROMPT_BLOCK = """\

@mcp.prompt()
def summarize_note(title: str) -> str:
    \"\"\"Summarize a specific note in exactly 3 bullets.

    The user invokes this as a slash command like:
        /summarize_note title="Q3-plan"
    The host substitutes the title parameter and uses the returned string as the LLM message.

    Args:
        title: The note title to summarize.
    \"\"\"
    body = NOTES.get(title, "[note not found]")
    return f\"\"\"Summarize the following note in exactly 3 bullet points.
Be concise; one sentence per bullet.

Note title: {title}

Note body:
{body}\"\"\"


if __name__ == "__main__":
    mcp.run()
"""

# Replace the placeholder line with the actual prompt + main block
existing = SERVER_PATH.read_text()
new_content = existing.replace(
    "# Prompt added in next step\n",
    PROMPT_BLOCK,
    1,
)
SERVER_PATH.write_text(new_content)
print(f"Final {SERVER_PATH}: {SERVER_PATH.stat().st_size} bytes, "
      f"{len(SERVER_PATH.read_text().splitlines())} lines")
print("\n--- File outline (function/decorator lines): ---")
for line in SERVER_PATH.read_text().splitlines():
    if "@mcp" in line or line.startswith("def ") or line.startswith("if __name"):
        print(f"  {line}")

## Step 4 — Test with the MCP Inspector

The [MCP Inspector](https://github.com/modelcontextprotocol/inspector) is the canonical development tool. Launch it pointing at your server, and it opens a browser UI at `localhost:5173` that lets you:
- See the server's introspected capabilities (tools, resources, prompts)
- Invoke each tool with a form-driven UI
- Read JSON-RPC messages in real time

The launch command is `fastmcp dev inspector notes_server.py` (FastMCP 3.x renamed it from `fastmcp dev`). In a Jupyter notebook, we launch it as a subprocess; **you must terminate the subprocess** before the next cell runs (Jupyter doesn't auto-clean subprocesses).

**Note**: this cell starts the Inspector in the background. Open `http://localhost:5173` in your browser to use it. When done, run the cleanup cell that follows to terminate the subprocess.

Three things to do in the Inspector:
1. Click the **Tools** tab. You should see `create_note`, `get_note`, `list_notes` with their docstrings as descriptions. Click any tool; see the JSON Schema derived from the type hints.
2. Click **create_note**, fill in title=`Q3-plan` and body=`Ship MCP module`, click **Run Tool**. Read the result in the right panel.
3. Click the **Resources** tab. You should see both `notes://all` and `notes://{title}`. Click `notes://{title}` and resolve it with `title=Q3-plan`; you'll get the body.

In [ ]:
import subprocess
import time

# Launch the MCP Inspector as a background subprocess
# (commented out by default so notebook reruns don't accidentally spawn duplicates)
# Uncomment to actually launch:
#
# inspector_proc = subprocess.Popen(
#     ["fastmcp", "dev", "inspector", str(SERVER_PATH)],
#     stdout=subprocess.PIPE,
#     stderr=subprocess.PIPE,
# )
# print(f"Inspector launched as PID {inspector_proc.pid}")
# print("Open http://localhost:5173 in your browser")
# time.sleep(3)  # give it a moment to start

print("⚠ Inspector subprocess launch is commented out by default to keep this notebook")
print("  re-runnable without leaking processes. Uncomment the block above to actually")
print("  launch the Inspector, then run the cleanup cell after you're done exploring.")
print()
print("Equivalent CLI invocation in your terminal:")
print(f"  fastmcp dev inspector {SERVER_PATH.name}")
print()
print("Then open http://localhost:5173 to interact with the server visually.")

In [ ]:
# Cleanup cell — terminate the Inspector subprocess if you launched it above
# inspector_proc.terminate()
# inspector_proc.wait(timeout=5)
# print("Inspector subprocess terminated")
print("(Cleanup cell — uncomment if you launched the Inspector above)")

## Step 5 — Test with the Python client (`fastmcp.Client`)

The Inspector is for humans. For programmatic testing — and for the agent code that'll consume your server in production — you use `fastmcp.Client`.

The client is an async context manager. Pass the server's path (or HTTP URL) and the client handles transport selection, initialize, capability listing, and shutdown.

We'll exercise three patterns:
1. List capabilities — `list_tools`, `list_resources`, `list_prompts`
2. Call each tool with realistic arguments
3. Read a templated resource

Run this cell after Step 1-3 have written the server file. The client spawns the server as a subprocess; you don't need to start it separately.

**A subtle distinction**: `client.list_resources()` returns *concrete* resources (the static `notes://all`); templated resources like `notes://{title}` are returned by `client.list_resource_templates()`. You can still *read* the templated resource by substituting the URI parameter, as the cell below does — the distinction is only in the listing API.

In [ ]:
import asyncio
from fastmcp import Client


async def exercise_server() -> dict:
    """Connect to the notes server and exercise every primitive."""
    results = {}
    async with Client(str(SERVER_PATH)) as client:
        # List capabilities
        tools = await client.list_tools()
        resources = await client.list_resources()
        prompts = await client.list_prompts()
        results["capability_counts"] = {
            "tools": len(tools),
            "resources": len(resources),
            "prompts": len(prompts),
        }
        results["tool_names"] = [t.name for t in tools]

        # Call each tool with realistic arguments
        result1 = await client.call_tool(
            "create_note",
            {"title": "Q3-plan", "body": "Ship MCP Module 1+2 in Batch 43."},
        )
        result2 = await client.call_tool(
            "create_note",
            {"title": "Q4-plan", "body": "Ship Module 3 next."},
        )
        result3 = await client.call_tool("get_note", {"title": "Q3-plan"})
        result4 = await client.call_tool("list_notes", {})

        results["call_results"] = {
            "create_Q3": result1.data,
            "create_Q4": result2.data,
            "get_Q3": result3.data,
            "list_all": result4.data,
        }

        # Read a templated resource
        templated = await client.read_resource("notes://Q3-plan")
        results["templated_resource"] = templated[0].text if templated else None

    return results


# Jupyter supports top-level await in cells; no asyncio.run() needed.
results = await exercise_server()
print("=== Server exercised programmatically ===\n")
print(f"Capability counts: {results['capability_counts']}")
print(f"Tool names: {results['tool_names']}")
print()
print("Tool call results:")
for k, v in results["call_results"].items():
    print(f"  {k}: {v}")
print()
print("Templated resource 'notes://Q3-plan':")
print(f"  {results['templated_resource']}")

## Step 6 — Deliberately-broken tools (the four schema-inference failure modes)

This is the high-leverage step. We'll write four broken versions of `create_note`, observe what each one does wrong, and fix it. These are the patterns that catch first-time MCP server authors most often.

**Failure mode 1: Vague docstring.** The LLM can't tell what your tool does, so it picks the wrong one.

**Failure mode 2: Missing parameter description.** FastMCP infers the type but not the *meaning*. The LLM doesn't know what `body` is supposed to contain.

**Failure mode 3: Wrong type narrowing.** Declaring `count: int` when you should have declared `count: Literal[1, 2, 3]` — the schema is over-permissive.

**Failure mode 4: Side-effectful tool without idempotency key.** Network or client retries duplicate state. Mentioned briefly here; deep-dive in [Path 03 Pattern 5 (Retry policies)](../../learning-paths/03-multi-agent-systems/patterns/05-retry-policies.md) and Module 4 (future).

For each mode, we'll write a deliberately-broken function, register it temporarily on a fresh `FastMCP` instance, introspect the generated schema, and contrast with a fixed version.

In [ ]:
from fastmcp import FastMCP

# Failure mode 1: vague docstring
demo_bad = FastMCP("demo-bad")


@demo_bad.tool()
def create(title: str, body: str) -> dict:
    """creates"""
    return {"status": "created", "title": title}


# Failure mode 1: FIXED — descriptive name + docstring
demo_good = FastMCP("demo-good")


@demo_good.tool()
def create_note_v2(title: str, body: str) -> dict:
    """Create a note with the given title and body. Fails if a note with
    the same title already exists.

    Args:
        title: The unique note title.
        body: The note body content; no length limit.

    Returns:
        A dict with 'status' (created|exists) and 'title'.
    """
    return {"status": "created", "title": title}


async def show_tool_schemas(server: FastMCP, label: str) -> None:
    """Connect to a FastMCP server and print its tool schemas."""
    # FastMCP's in-memory transport: pass the server object directly
    async with Client(server) as client:
        tools = await client.list_tools()
        print(f"=== {label} ===")
        for tool in tools:
            print(f"  Name: {tool.name}")
            print(f"  Description: {tool.description!r}")
            print("  Input schema:")
            for k, v in (tool.inputSchema or {}).items():
                print(f"    {k}: {v}")
        print()


await show_tool_schemas(demo_bad, "FAILURE MODE 1: Vague docstring")
await show_tool_schemas(demo_good, "FIXED: Descriptive docstring + parameter docs")

In [ ]:
from typing import Literal

# Failure mode 3: type too permissive
demo_loose = FastMCP("demo-loose")


@demo_loose.tool()
def set_priority_loose(title: str, level: int) -> dict:
    """Set priority on a note. level should be 1, 2, or 3.

    Args:
        title: The note title.
        level: Priority level. Must be 1, 2, or 3.
    """
    return {"status": "ok", "title": title, "level": level}


# Failure mode 3: FIXED — Literal narrows the schema
demo_tight = FastMCP("demo-tight")


@demo_tight.tool()
def set_priority_tight(title: str, level: Literal[1, 2, 3]) -> dict:
    """Set priority on a note. level must be 1, 2, or 3.

    Args:
        title: The note title.
        level: Priority level constrained to 1, 2, or 3.
    """
    return {"status": "ok", "title": title, "level": level}


await show_tool_schemas(demo_loose, "FAILURE MODE 3: int allows any integer")
await show_tool_schemas(demo_tight, "FIXED: Literal[1,2,3] constrains the schema")

print("Notice the inputSchema for level in the FIXED version has an 'enum' constraint.")
print("The LLM will only generate level=1, 2, or 3. The loose version would accept 999.")

## Step 7 — Streamable HTTP deployment

stdio is for local development. **Streamable HTTP is the production default** since the 2026 MCP spec.

Two changes from the stdio setup:
1. **`mcp.run(transport="streamable-http", host="0.0.0.0", port=8000)`** — switch the transport.
2. **Add an auth provider.** A network-exposed server with no auth is a network-exposed dict; anyone who can reach the port can read and modify your notes.

FastMCP 3.0 ships pluggable auth providers per [firecrawl.dev April 2026](https://www.firecrawl.dev/blog/fastmcp-tutorial-building-mcp-servers-python). The simplest is a token-based provider for internal services. Production-public deployments escalate to full OAuth 2.1, which is Path 07 territory.

For this lab, we'll write the HTTP-transport version of `notes_server.py` and demonstrate token-based auth conceptually. We do NOT bind to `0.0.0.0` from this notebook (that would expose your dev machine to the network); we show the pattern only.

In [ ]:
HTTP_SERVER_PATH = LAB_DIR / "notes_server_http.py"

HTTP_SERVER_CODE = """\
\"\"\"Lab 25 — Streamable HTTP variant of the notes server with token auth.\"\"\"
import os
from fastmcp import FastMCP

mcp = FastMCP("notes-server-http")
NOTES: dict[str, str] = {}

# Read the expected bearer token from environment (do NOT hardcode)
EXPECTED_TOKEN = os.environ.get("MCP_BEARER_TOKEN", "")


@mcp.tool()
def create_note(title: str, body: str) -> dict:
    \"\"\"Create a note with the given title and body.

    Args:
        title: The unique note title.
        body: The note body.

    Returns:
        A dict with 'status' (created|exists) and 'title'.
    \"\"\"
    if title in NOTES:
        return {"status": "exists", "title": title}
    NOTES[title] = body
    return {"status": "created", "title": title}


@mcp.tool()
def list_notes() -> list[str]:
    \"\"\"List all note titles.\"\"\"
    return sorted(NOTES.keys())


if __name__ == "__main__":
    # Production: bind to 0.0.0.0 for external access; use 127.0.0.1 for local-only.
    # Production: add full OAuth 2.1 provider, not just bearer-token check.
    # Production: put TLS termination at your reverse proxy (nginx, Caddy).
    mcp.run(
        transport="streamable-http",
        host="127.0.0.1",  # local-only for this lab
        port=8000,
    )
"""

HTTP_SERVER_PATH.write_text(HTTP_SERVER_CODE)
print(f"Wrote {HTTP_SERVER_PATH} ({HTTP_SERVER_PATH.stat().st_size} bytes)")
print()
print("To run the Streamable HTTP server:")
print("  export MCP_BEARER_TOKEN=\"$(openssl rand -hex 32)\"")
print(f"  python {HTTP_SERVER_PATH.name}")
print()
print("Then connect from a separate terminal:")
print("  curl -X POST http://127.0.0.1:8000/mcp/v1 \\")
print("    -H 'Authorization: Bearer <token>' \\")
print("    -H 'Content-Type: application/json' \\")
print("    -d \'{\"jsonrpc\":\"2.0\",\"id\":1,\"method\":\"tools/list\"}\'")
print()
print("⚠ This lab does NOT spawn the HTTP server inline — that would tie up the port")
print("  and Jupyter kernel. Run it manually in a separate terminal to exercise it.")

## Step 8 — Stretch: persistence, more CRUD, rate limits

The lab's core is done at Step 7. This step is stretch material — directions worth exploring on your own:

**Persistence with SQLite.** Replace the `NOTES: dict[str, str]` with a SQLite-backed store. The tool functions look identical; only the storage layer changes. This is the production-default pattern.

**More CRUD: `delete_note` and `update_note`.** Mirror the `create_note` shape. `update_note(title, body)` should return `status="not_found"` if the note doesn't exist. `delete_note(title)` should return `status="not_found"` or `status="deleted"`.

**Rate limiting.** A decorator-based rate limit (e.g., `slowapi` for Starlette-style apps) on the Streamable HTTP server. The pattern is mentioned briefly in [Path 03 Pattern 5 (Retry policies)](../../learning-paths/03-multi-agent-systems/patterns/05-retry-policies.md). Module 4 (future) covers MCP-specific rate-limiting strategies.

**Connect to Claude Desktop.** Add the server to `~/Library/Application Support/Claude/claude_desktop_config.json` (macOS) or `%APPDATA%\\Claude\\claude_desktop_config.json` (Windows). Three-line config; the [MCP Python SDK README](https://github.com/modelcontextprotocol/python-sdk) has the exact format.

None of these is required to finish the lab. They're the natural next experiments once the core mechanics are clear.

## What you've built

You now have:
- 📄 `notes_server.py` — a stdio MCP server with 3 tools, 2 resources, 1 prompt
- 📄 `notes_server_http.py` — the Streamable HTTP variant with token-based auth
- A working understanding of FastMCP's three decorators, type-hint schema inference, the MCP Inspector workflow, and the four most common schema-inference failure modes

## Where this goes next

This server is the substrate for the rest of [Path 04](../../learning-paths/04-tool-protocols-mcp-a2a/):

- **Module 3 — Building an MCP client**: future lab; will use this server as a target. Covers tool discovery, error handling, and the MCP Server Cards spec (H2 2026)
- **Module 4 — MCP security threat model**: future lab; the [arxiv:2601.10955](https://arxiv.org/abs/2601.10955) resource-amplification attack walked through against a server like this one
- **Module 5-7 — A2A**: the agent-to-agent half of Path 04; composes with MCP servers like this one

## Test yourself

Once you've worked through the notebook, take the [MCP foundations and server quiz](../../quizzes/foundations/mcp-foundations-and-server.md) — 8 questions covering both concept pages and this lab.